In [1]:
import numpy as np
from scipy import stats
from scipy.optimize import brentq

def power_continuous(n_per_arm, effect_size, sd, alpha=0.05, allocation_ratio=1.0):
    n1 = n_per_arm
    n2 = n_per_arm * allocation_ratio
    df = n1 + n2 - 2
    ncp = effect_size / (sd * np.sqrt(1/n1 + 1/n2))
    t_crit = stats.t.ppf(1 - alpha / 2, df)
    return 1 - stats.nct.cdf(t_crit, df, ncp) + stats.nct.cdf(-t_crit, df, ncp)

In [ ]:
def sample_size_continuous(effect_size, sd, alpha=0.05, power=0.8, allocation_ratio=1.0):
    def f(n):
        return power_continuous(n, effect_size, sd, alpha, allocation_ratio) - power

    lo = 2.0
    if f(lo) >= 0:
        return lo  

    hi = 4.0
    f_hi = f(hi)
    while not (np.isfinite(f_hi) and f_hi > 0):
        hi *= 1.5 if np.isfinite(f_hi) else 1.05
        f_hi = f(hi)
        if hi > 1e7:
            raise RuntimeError("could not bracket a solution, check inputs")

    return brentq(f, lo, hi)

In [ ]:
n_needed = sample_size_continuous(effect_size=0.5, sd=1.0, alpha=0.05, power=0.8)
print(n_needed)  

63.76561018989429
